# Exploring the Simulated Data

The data is in `../datasets/series0/run*-*/fiber_position.txt`.

In [22]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm 
import torch 
import torch.nn as nn
import sys

In [57]:
# Add torch gpu 
device = torch.device('mps')

In [58]:
device

device(type='mps')

Let's start by getting a list of paths to every `fiber_position.txt` file. 

In [59]:
base_path = '../datasets/series0/'
# now we want to get a list of all the folders with the name run*
run_folders = [f for f in os.listdir(base_path) if os.path.isdir(os.path.join(base_path, f)) and f.startswith('run')]
run_folders

['run0000-0001',
 'run0000-0006',
 'run0000-0008',
 'run0000-0009',
 'run0000-0007',
 'run0000-0000',
 'run0000-0013',
 'run0000-0014',
 'run0000-0015',
 'run0000-0012',
 'run0000',
 'run0000-0005',
 'run0000-0002',
 'run0000-0003',
 'run0000-0004',
 'run0000-0017',
 'run0000-0010',
 'run0000-0011',
 'run0000-0016',
 'run0000-0018']

In [60]:
# now we can import each of the `fiber_position.txt` files for each of the `run_folders`

# we can use the `glob` module to get all the files in a directory that match a pattern
import glob

path_list = glob.glob(base_path + 'run*/fibers.txt')
path_list

['../datasets/series0/run0000-0001/fibers.txt',
 '../datasets/series0/run0000-0006/fibers.txt',
 '../datasets/series0/run0000-0008/fibers.txt',
 '../datasets/series0/run0000-0009/fibers.txt',
 '../datasets/series0/run0000-0007/fibers.txt',
 '../datasets/series0/run0000-0000/fibers.txt',
 '../datasets/series0/run0000-0013/fibers.txt',
 '../datasets/series0/run0000-0014/fibers.txt',
 '../datasets/series0/run0000-0015/fibers.txt',
 '../datasets/series0/run0000-0012/fibers.txt',
 '../datasets/series0/run0000/fibers.txt',
 '../datasets/series0/run0000-0005/fibers.txt',
 '../datasets/series0/run0000-0002/fibers.txt',
 '../datasets/series0/run0000-0003/fibers.txt',
 '../datasets/series0/run0000-0004/fibers.txt',
 '../datasets/series0/run0000-0017/fibers.txt',
 '../datasets/series0/run0000-0010/fibers.txt',
 '../datasets/series0/run0000-0011/fibers.txt',
 '../datasets/series0/run0000-0016/fibers.txt',
 '../datasets/series0/run0000-0018/fibers.txt']

In [61]:
import pandas as pd

def find_frames(file_path):
    frames = []
    with open(file_path, 'r') as f:
        lines = f.readlines()
    current_frame = None
    for i, line in enumerate(lines):
        if '% frame' in line:
            current_frame = int(line.split()[-1])
        elif 'organizer' in line:
            start_line = i + 1
            while start_line < len(lines) and not lines[start_line].startswith('%'):
                frames.append((current_frame, start_line))
                start_line += 1
    return frames

def read_frame_into_dataframe(file_path, frame, start_line):
    column_names = ['class', 'identity', 'length', 'posX', 'posY', 'dirX', 'dirY', 'endToEnd', 'cosinus', 'organizer']
    data = pd.read_csv(file_path, 
                       skiprows=start_line, 
                       nrows=1, 
                       delim_whitespace=True, 
                       names=column_names)
    data['frame'] = frame
    return data

def load_frames_from_text_file(file_path):
    print('finding frames...')
    frames = find_frames(file_path)
    print("done finding frames")

    print('\nreading frames...')
    dataframes = [read_frame_into_dataframe(file_path, frame, start_line) for frame, start_line in tqdm(frames)]
    print('done reading frames')
    print('')
    data = pd.concat(dataframes, ignore_index=True)
    return data


In [62]:
data_1 = load_frames_from_text_file(path_list[0])

finding frames...
done finding frames

reading frames...


100%|██████████| 24949/24949 [01:35<00:00, 260.67it/s]


done reading frames



In [63]:
path_list[0]

'../datasets/series0/run0000-0001/fibers.txt'

In [64]:
import pdb

In [65]:
# Function to get particle state at a given timestep from one of the dataframes 
# produced by `load_frames_from_text_file`
def get_particle_states_at_timestep(df, timestep, state_data_only=True):
    df_timestep = df[df['frame'] == timestep]
    df_timestep.set_index('identity', inplace=True)
    num_particles = df['identity'].max()
    
    # Create a dataframe with an index for every particle
    all_particles = pd.DataFrame(index=pd.RangeIndex(start=1, stop=num_particles+1, step=1))
    
    # Join the data from the selected timestep to all_particles
    all_particles = all_particles.join(df_timestep)
    
    # Drop unnecessary columns
    all_particles.drop(columns='frame', inplace=True)
    
    # Replace NaNs with a sensible value or leave them as NaNs
    all_particles.fillna(np.nan, inplace=True)

    if state_data_only:
        all_particles = all_particles[['posX', 'posY', 'dirX', 'dirY']]
    
    return all_particles.values

In [66]:
state_0 = get_particle_states_at_timestep(data_1, 0)

In [67]:
state_0[:10, :] # class, length, posX, posY, dirX, dirY, endToEnd, cosinus, organizer 
               # now: posX, posY, dirX, dirY

array([[ -4.7754,   3.9449,  -0.2544,  -0.9671],
       [ -0.8557,  -2.2117,   0.9088,   0.4171],
       [  9.7576,  10.2197,  -0.7538,   0.6571],
       [  4.548 ,  -4.4291,  -0.6034,   0.7974],
       [  6.0872,  13.7428,   0.3054,  -0.9522],
       [ -2.6815,  10.6676,  -0.9789,   0.2042],
       [ 14.1094,   6.3107,  -0.6293,   0.7772],
       [ -3.0793, -17.008 ,   0.9425,   0.3341],
       [ -3.7658,  10.6605,   0.7007,  -0.7135],
       [  4.9935,  -9.5587,  -0.9614,   0.275 ]])

# Multi-Head Attention

In [75]:
num_latents = 160 # arbitrary
num_particles = state_0.shape[0] # 409
dim_observations = 4 # posX, posY, dirX, dirY


In [76]:
## Setting up the MHA 
embed_dim = num_latents + dim_observations   # Key and query dimension expected in the input.
key_dim = embed_dim     # Expected dimension of the keys(?)
val_dim = embed_dim     # Expected dimension of the 
num_heads = 1   # Number of heads in the MHA.
multihead_attn = nn.MultiheadAttention(embed_dim, num_heads, kdim=key_dim, vdim=val_dim, bias=False) # value/output dimension should be the same as `embed_dim` by default

In [77]:
num_batch = 1
start_tensor = torch.zeros([num_particles, num_batch, embed_dim])

In [78]:
multihead_attn(start_tensor, start_tensor, start_tensor)[0].shape

torch.Size([409, 1, 164])

In [79]:
start_tensor.shape

torch.Size([409, 1, 164])

## Training Loop 

In [80]:
# for loop through each frame 
num_frames = 10
dt = 0.1
num_epochs = 50000

loss_history = []

# define the adam optimizer 
optimizer = torch.optim.Adam(multihead_attn.parameters(), lr=0.001)

pbar = tqdm(range(num_epochs))
for epoch in pbar:
    loss = 0
    start_tensor = torch.zeros([num_particles, num_batch, embed_dim])
    # get the first observation on the particles from data_1
    obs_0 = get_particle_states_at_timestep(data_1, 0)
    start_tensor[:, :, :4] = torch.tensor(obs_0)[:, None, :]

    state_tensor_history = [start_tensor]
    for i in range(1, num_frames):
        obs_i = get_particle_states_at_timestep(data_1, i)
        # start_tensor[:, :, 4:] = torch.tensor(obs_i)[:, None, :]

        # get dX/dt
        d_X = dt * multihead_attn(state_tensor_history[-1], start_tensor, start_tensor)[0]

        # update the state tensor (no in-place operations)
        start_tensor_new = state_tensor_history[-1] + d_X

        # append to state tensor history
        state_tensor_history.append(start_tensor_new)

        new_loss = torch.mean(torch.abs(state_tensor_history[-1][:, :, :4] - torch.tensor(obs_i)[:, None, :])**2)
        loss += new_loss

    loss.backward()
    optimizer.step()

    optimizer.zero_grad()
    # append to loss history 
    loss_history.append(loss.detach().numpy())
    # print(f'Epoch: {epoch}, Loss: {loss.detach().numpy()}')
    # update pbar 
    pbar.set_description(f'Epoch: {epoch}, Loss: {loss.detach().numpy()}')



Epoch: 354, Loss: 35.794656529454336:   1%|          | 355/50000 [00:10<23:18, 35.50it/s]

In [33]:
start_tensor.shape

torch.Size([409, 1, 7])